<a href="https://colab.research.google.com/github/rcabell/wrf_hydro_training/blob/master/plot_streamflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Streamflow observations and model scenarios plotting

In [ ]:
#@title Initialization
import os, sys
sys.stderr = open('/dev/null', 'w')
orig_stdout = sys.stdout
sys.stdout = open('/dev/null', 'w') # disable install output

DEVELOPMENT_MODE = False
# uncomment for colab
!pip install jupyter_bokeh --no-deps

if DEVELOPMENT_MODE:
    sys.stdout = orig_stdout
    ROOT_DIR = f"/Volumes/d1/gaydos/git/ral-idwr-notebooks"
    USE_MULTIPROCESSING = False
else:
    ROOT_DIR = f"/content/"
    USE_MULTIPROCESSING = True

    UTIL_ID = "1s3eEAA5GSNerrEIbWQpETxFlCGg8UHjs"
    DATA_ID = "1p6l_WimaGML-2KO1CBnnc9XxZR8YZLBk"
    DATA_ZIP_ID = "1rYWLipXl0A8_hiGLdnNZRj9k9FU0DDzD"
    import gdown, zipfile
    if not os.path.exists(f"{ROOT_DIR}/util"):
      gdown.download_folder(id=UTIL_ID, quiet=False, use_cookies=False)
    if not os.path.exists(f"{ROOT_DIR}/data"):
      gdown.download(id=DATA_ZIP_ID, output="data.zip")
      gdown.extractall('data.zip')

import bokeh.plotting
import bokeh.io
import panel as pn

pn.extension('plotly')

bokeh.io.output_notebook()

helper_script_dir = f'{ROOT_DIR}/util/'

# Add the directory to Python's system path if it's not already there
print("Setting module path")
if helper_script_dir not in sys.path:
    sys.path.append(helper_script_dir)
    print(f"'{helper_script_dir}' added to sys.path.")

In [ ]:
#@title Load modules
# Remove development modules from sys.modules to force a reload
for m in ['util', 'streamflow']:
  if m in sys.modules:
    print(f"Unloading {m} module")
    del sys.modules[m]

import pandas as pd
import plotly.express as px
from util import DataHelper, CSV, RDATA, PARQUET
from streamflow import StreamflowHelper

dataObj = DataHelper(ROOT_DIR)
streamObj = StreamflowHelper()
streamObj.set_gage_metadata(f'{ROOT_DIR}/data/gage_meta.txt', delim='\t')


In [ ]:
#@title Load data
# try to load data in parallel
import multiprocessing

datasets = {
    'obs': {'name': "Observations", 'fname': "obsStrData_NWIS_USB_2000_2022_IV.parquet"}
}

for i in range(1,16):
  fname = f"chan_BiasCorrect404_S{i}_USB.parquet"
  if not os.path.exists(f'{ROOT_DIR}/data/streamflow/{fname}'):
    continue
  datasets[f'S{i}'] = {'name': f"Scenario-{i}", 'fname': fname}

def runner(fname):
  print(f"Loading {fname}")
  ds = dataObj.get_data(f'{ROOT_DIR}/data/streamflow/{fname}', type=PARQUET)
  return ds

args = [datasets[k]['fname'] for k in datasets]

if USE_MULTIPROCESSING:
  with multiprocessing.Pool() as pool:
    results = pool.map(runner, args)

    for i, k in enumerate(datasets):
      datasets[k]['df'] = results[i]
else:
  for a in args:
    ds = runner(a)
    for k in datasets:
      if datasets[k]['fname'] == a:
        datasets[k]['df'] = ds

# Other datasets
stats_df = pd.read_parquet(f'{ROOT_DIR}/data/streamflow/streamflow_scenario_comparisons.parquet')
wy_df = pd.read_csv(f'{ROOT_DIR}/data/streamflow/modChrtout_wy_USB.tsv',delimiter='\t')

# set up ID list and scenarios for plotting
unique_station_ids = stats_df['site_no'].unique()
scenario_names = [s.split("_")[-1] for s in stats_df['tag_comp'].unique()]

station_selector = streamObj.get_station_selector(station_ids=unique_station_ids)


In [ ]:
#@title Create streamflow plots
# set up ID list and scenarios for plotting

scenarios = {}
names = {}
snames = []

for d in datasets:
  scenarios[d] = datasets[d]['df']
  names[d] = datasets[d]['name']
  if d != 'obs':
    snames.append(d)

scenario_selector = streamObj.get_scenario_selector(snames, default_scenario=list(snames)[2:4], multiple=True)

period_selector = pn.widgets.Select(
    name="Select Period",
    options={'Month': streamObj.MONTH, 'Day': streamObj.DAY},
    value=streamObj.MONTH # Default to Monthly
)

units_selector = pn.widgets.Select(
    name="Select Units",
    options={'KAF': "kaf", 'CFS': "cfs"},
    value="kaf" # Default to kaf
)

@pn.depends(station_selector.param.value, period_selector.param.value, scenario_selector.param.value, units_selector.param.value)
def update_plot(station_id_selected, selected_period, selected_scenarios, selected_units):
    # Filter the scenarios and names based on the selection
    filtered_scenarios = {key: scenarios[key] for key in selected_scenarios}
    # Always include 'obs' in the names for plotting
    filtered_names = {'obs': names['obs']}
    filtered_names.update({key: names[key] for key in selected_scenarios})

    # Get data for the selected station, passing filtered scenarios and names
    selected_data = streamObj.get_reach(station_id_selected, scenarios['obs'], filtered_scenarios)
    fig = streamObj.get_streamflow_plot(station_id_selected, selected_data, filtered_names, period=selected_period, units=selected_units)
    return fig


widgets = pn.Column(
    pn.Row(
        station_selector,
        period_selector
    ),
    pn.Row(
        scenario_selector,
        units_selector
    )
)

pn.Column(pn.pane.Plotly(update_plot, width=1200, height=600), widgets)

In [ ]:
#@title Create accumulated flow plots
scenario_selector = streamObj.get_scenario_selector(snames, default_scenario=['S3','S4'], multiple=True)

@pn.depends(station_selector.param.value, scenario_selector.param.value)
def update_accum_plot(id, selected_scenarios):
    return streamObj.get_accum_streamflow_plot(id, scenarios, selected_scenarios)

widgets_boxplot = pn.Row(
    station_selector,
    scenario_selector
)

pn.Column(pn.pane.Plotly(update_accum_plot, width=1000, height=600), widgets_boxplot)

In [ ]:
#@title Create stats ribbon plots
scenario_selector = streamObj.get_scenario_selector(scenario_names, default_scenario=scenario_names[0], multiple=False)

@pn.depends(station_selector.param.value, scenario_selector.param.value)
def update_stats_plot(id, selected_scenario):
  fig = streamObj.get_delta_plot(id, stats_df, selected_scenario)
  return fig


widgets = pn.Row(
        station_selector,
        scenario_selector
    )

pn.Column(pn.pane.Plotly(update_stats_plot, width=1200, height=600), widgets)




In [ ]:
#@title Create box plots
scenario_selector = streamObj.get_scenario_selector(scenario_names, default_scenario=['S4','S5','S6','S7','S8','S9','S10','S13'], multiple=True)

@pn.depends(station_selector.param.value, scenario_selector.param.value)
def update_boxplot(id, selected_scenarios):
    return streamObj.get_streamflow_boxplot(id, wy_df, selected_scenarios)

widgets_boxplot = pn.Row(
    station_selector,
    scenario_selector
)

pn.Column(pn.pane.Plotly(update_boxplot, width=1000, height=600), widgets_boxplot)